# 67) Post-Hoc Analizi Nedir?
ANOVA bize sadece **"en az bir grup farklı"** dedi (Konu 66'daki sonuç) ama **hangi grup(lar)ın** birbirinden farklı olduğunu söylemedi. 
Post-Hoc ("sonradan" anlamına gelir, Latince) testler, ANOVA anlamlı çıktıktan **sonra**, grupları ikişer ikişer karşılaştırarak bu soruyu cevaplar.

## Neden Direkt İkili T Testleri Yapmıyoruz?
Hatırlarsak, Konu (FamilyWise Error Rate)'de bunun tehlikesini görmüştük çok sayıda ikili karşılaştırma, Alpha Inflation riskini doğurur. 
Post-Hoc testler, bu riski **kontrol ederek** ikili karşılaştırmalar yapan **özel yöntemlerdir** — yani hem "hangisi farklı" sorusunu cevaplıyor hem de çoklu test riskini otomatik olarak düzeltiyor.

## En Yaygın Post-Hoc Yöntemleri

**1. Tukey HSD (Honestly Significant Difference):** En sık kullanılan yöntem. Tüm grup çiftlerini karşılaştırır, Alpha Inflation'ı otomatik kontrol eder. Varyansların homojen olduğu durumlarda idealdir.

**2. Bonferroni Post-Hoc:** Konu (FWER)'de öğrendiğimiz Bonferroni düzeltmesinin, ikili karşılaştırmalara uygulanmış hali. Tukey'den daha **katı/muhafazakar**dır.

**3. Games-Howell:** Varyanslar **homojen değilse** (Levene testi 
reddedilmişse) tercih edilir — Tukey'nin varyans eşitliği varsayımına 
ihtiyaç duymaz.

## Hangisini Seçmeli?
- Varyanslar homojense (bizim örneğimizdeki gibi, Levene p=0.0746>0.05) → **Tukey HSD**
- Varyanslar homojen değilse → **Games-Howell**

## Python'da Kullanımı
```python
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sonuc = pairwise_tukeyhsd(endog=tum_veri, groups=grup_etiketleri, alpha=0.05)
print(sonuc)
```
Dikkat: `pairwise_tukeyhsd`, bizim ayrı ayrı tuttuğumuz 3 diziyi değil, 
**tek bir birleşik veri dizisi + hangi grubun hangisi olduğunu belirten 
etiket dizisi** ister — verileri bu formata dönüştürmemiz gerekecek.

## Sonucun Yorumlanması
Tukey testinin çıktısı, her ikili karşılaştırma için ayrı bir p-değeri/
"reject" (True/False) sütunu verir — `reject=True` olan çiftler, 
istatistiksel olarak anlamlı şekilde farklıdır.

In [1]:
import numpy as np

np.random.seed(42)
np.set_printoptions(suppress=False)
# Senaryo: 4 farklı destek kanalından gelen müşteri memnuniyet puanları (1-100)
telefon = np.random.normal(loc=72, scale=8, size=40)
canli_sohbet = np.random.normal(loc=80, scale=7, size=40)
email = np.random.normal(loc=68, scale=9, size=40)
sosyal_medya = np.random.normal(loc=75, scale=10, size=40)

from scipy import stats

# H0: 4 destek kanalının ortalama memnuniyet puanı aynıdır.
# H1: En az bir kanalın ortalama memnuniyet puanı diğerlerinden farklıdır.

f_degeri, anova_p = stats.f_oneway(telefon, canli_sohbet, email, sosyal_medya)
print(f"F istatistiği: {f_degeri}")
print(f"P-değeri: {anova_p}")

import pandas as pd

# Önce "geniş format" (wide format) her kanal ayrı bir sütun
df_genis = pd.DataFrame({
    'Telefon': telefon,
    'Canlı Sohbet': canli_sohbet,
    'Email': email,
    'Sosyal Medya': sosyal_medya
})
# melt() ile "uzun format"a (long format) çeviriyoruz
df_uzun = df_genis.melt(var_name='Kanal', value_name='Memnuniyet')

from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_sonuc = pairwise_tukeyhsd(endog=df_uzun['Memnuniyet'], groups=df_uzun['Kanal'], alpha=0.05)
print(tukey_sonuc)

F istatistiği: 16.503894051636372
P-değeri: 2.303586805368447e-09
       Multiple Comparison of Means - Tukey HSD, FWER=0.05       
   group1       group2    meandiff p-adj   lower    upper  reject
-----------------------------------------------------------------
Canlı Sohbet        Email -11.7051    0.0 -16.3873 -7.0229   True
Canlı Sohbet Sosyal Medya  -5.1257 0.0258  -9.8079 -0.4435   True
Canlı Sohbet      Telefon  -9.5458    0.0  -14.228 -4.8636   True
       Email Sosyal Medya   6.5794  0.002   1.8972 11.2616   True
       Email      Telefon   2.1593 0.6292  -2.5228  6.8415  False
Sosyal Medya      Telefon  -4.4201 0.0718  -9.1023  0.2621  False
-----------------------------------------------------------------


### Sonuç
Tukey HSD sonuçlarına göre: Canlı Sohbet kanalı (ort=79.80), en yüksek memnuniyet puanına sahip olup, Email (ort=68.09), Sosyal Medya (ort=74.67) ve Telefon (ort=70.25) kanallarından anlamlı şekilde daha yüksektir (sırasıyla 11.71, 5.13, ve 9.55 puan fark ile). Email kanalı ise Sosyal Medya kanalından anlamlı şekilde daha düşük memnuniyet almaktadır (6.58 puan fark). Email-Telefon ve Sosyal Medya-Telefon arasındaki farklar istatistiksel olarak anlamlı bulunmamıştır (p>0.05).